# NSynth relative pitch triplet demo

This notebook builds a small contrastive-style triplet sampler on NSynth validation clips. Each example is an interval made from two notes (start → target). Anchor and positive share the same interval and instrument family but start on different MIDI notes; the negative reuses the positive's start note and family with a different interval. Notes are trimmed to 400 ms with a cosine fade-out, and each 2-note clip is exactly 2.0 s.


In [89]:
import json
import math
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torchaudio
import torch.nn.functional as F

from IPython.display import Audio, display

%matplotlib inline
import matplotlib.pyplot as plt


In [90]:
# Point to the extracted NSynth validation split
nsynth_root = Path("/mnt/home/igriffith/ceph/datasets/nsynth")
split = "valid"
metadata_path = nsynth_root / f"nsynth-{split}" / "examples.json"
audio_dir = nsynth_root / f"nsynth-{split}" / "audio"

assert metadata_path.exists(), f"Missing metadata: {metadata_path}"
assert audio_dir.exists(), f"Missing audio dir: {audio_dir}"

with metadata_path.open() as f:
    metadata: Dict[str, Dict] = json.load(f)


print(f"Loaded {len(metadata):,} {split} examples from {metadata_path}")


Loaded 12,678 valid examples from /mnt/home/igriffith/ceph/datasets/nsynth/nsynth-valid/examples.json


In [91]:
# Build pitch / instrument-family indices for quick sampling
INSTRUMENT_FIELD = "instrument_family_str"  # use family to keep timbre consistent

pitch_to_ids: Dict[int, List[str]] = defaultdict(list)
pitch_family_to_ids: Dict[int, Dict[str, List[str]]] = defaultdict(lambda: defaultdict(list))
instrument_families = set()

for file_id, meta in metadata.items():
    pitch = int(meta["pitch"])
    family = meta[INSTRUMENT_FIELD]
    pitch_to_ids[pitch].append(file_id)
    pitch_family_to_ids[pitch][family].append(file_id)
    instrument_families.add(family)

available_pitches = sorted(pitch_to_ids)
instrument_families = sorted(instrument_families)
print(f"Found {len(available_pitches)} unique MIDI pitches and {len(instrument_families)} instrument families in {split} split")


Found 112 unique MIDI pitches and 10 instrument families in valid split


In [92]:
NOTE_DURATION_SEC = 0.5
TOTAL_CLIP_SEC = 2.0
MID_GAP_SEC = 0.4
LEAD_IN_SEC = 0.25  # silence before the first note
FADE_MS = 40.0
MIN_INTERVAL = 1
MAX_INTERVAL = 12
MIN_MIDI = 40
MAX_MIDI = 70


def load_audio(file_id: str) -> Tuple[torch.Tensor, int]:
    """Load an NSynth waveform and return mono audio plus sample rate."""
    audio_path = audio_dir / f"{file_id}.wav"
    waveform, sr = torchaudio.load(str(audio_path))
    if waveform.shape[0] > 1:
        waveform = waveform[0] # just take single channel
    return waveform.squeeze(0), sr


def crop_with_cosine_fade(
    audio: torch.Tensor,
    sr: int,
    target_sec: float = NOTE_DURATION_SEC,
    ramp_ms: float = FADE_MS,
) -> torch.Tensor:
    """Trim/pad to target duration and apply cosine fade-out at the end."""
    target_samples = int(sr * target_sec)

    if audio.numel() >= target_samples:
        audio = audio[:target_samples]
    else:
        pad = target_samples - audio.numel()
        audio = F.pad(audio, (0, pad))

    ramp_samples = int(sr * ramp_ms / 1000.0)
    if ramp_samples > 0 and target_samples >= ramp_samples:
        t = torch.linspace(0, math.pi, ramp_samples, device=audio.device)
        fade = 0.5 * (1 + torch.cos(t))  # 1 -> 0
        audio[-ramp_samples:] = audio[-ramp_samples:] * fade
    return audio


def choose_file_for_pitch_family(pitch: int, family: str, rng: random.Random) -> str:
    """Randomly pick a file ID for a given MIDI pitch and instrument family."""
    if pitch not in pitch_family_to_ids or family not in pitch_family_to_ids[pitch]:
        raise ValueError(f"Pitch {pitch} with family {family} not in split")
    return rng.choice(pitch_family_to_ids[pitch][family])


def families_supporting_interval(start_pitch: int, interval: int) -> List[str]:
    target_pitch = start_pitch + interval
    if target_pitch < MIN_MIDI or target_pitch > MAX_MIDI:
        return []
    start_fams = set(pitch_family_to_ids.get(start_pitch, {}))
    target_fams = set(pitch_family_to_ids.get(target_pitch, {}))
    return sorted(start_fams & target_fams)


def valid_interval_starts(interval: int, family: str) -> List[int]:
    """All starting pitches that have a partner pitch offset by `interval` for a family within the MIDI window."""
    starts: List[int] = []
    for p in available_pitches:
        if p < MIN_MIDI or p > MAX_MIDI:
            continue
        target = p + interval
        if target < MIN_MIDI or target > MAX_MIDI:
            continue
        if target in pitch_family_to_ids:
            if family in pitch_family_to_ids[p] and family in pitch_family_to_ids[target]:
                starts.append(p)
    return starts


def make_interval_clip(start_id: str, target_id: str) -> Tuple[torch.Tensor, int]:
    """Concatenate start and target notes with a gap, trimmed to 2s total."""
    start_audio, sr_start = load_audio(start_id)
    target_audio, sr_target = load_audio(target_id)

    if sr_start != sr_target:
        target_audio = torchaudio.functional.resample(
            target_audio.unsqueeze(0), sr_target, sr_start
        ).squeeze(0)
        sr = sr_start
    else:
        sr = sr_start

    start_audio = crop_with_cosine_fade(start_audio, sr)
    target_audio = crop_with_cosine_fade(target_audio, sr)

    lead_in = torch.zeros(int(sr * LEAD_IN_SEC))
    gap = torch.zeros(int(sr * MID_GAP_SEC))
    tail = max(0, int(sr * (TOTAL_CLIP_SEC - (LEAD_IN_SEC + 2 * NOTE_DURATION_SEC + MID_GAP_SEC))))
    tail_pad = torch.zeros(tail)

    clip = torch.cat([lead_in, start_audio, gap, target_audio, tail_pad])

    target_total = int(sr * TOTAL_CLIP_SEC)
    if clip.numel() > target_total:
        clip = clip[:target_total]
    elif clip.numel() < target_total:
        clip = torch.cat([clip, torch.zeros(target_total - clip.numel())])

    return clip, sr



In [93]:
def validate_interval(interval: int, name: str) -> None:
    if interval == 0 or abs(interval) < MIN_INTERVAL or abs(interval) > MAX_INTERVAL:
        raise ValueError(
            f"{name} must be a non-zero interval between ±{MIN_INTERVAL} and ±{MAX_INTERVAL} semitones; got {interval}"
        )


def build_interval_triplet(
    interval: int = 4,
    negative_interval: int | None = None,
    rng_seed: int | None = 0,
) -> Dict[str, Dict]:
    """Create anchor/positive/negative interval examples as file IDs.

    Anchor and positive share the same `interval` and instrument family but start on
    different MIDI notes. The negative shares the positive's start note + family but
    uses a different interval (still within ±12 semitones).
    """
    rng = random.Random(rng_seed)

    validate_interval(interval, "interval")
    if negative_interval is not None:
        validate_interval(negative_interval, "negative_interval")
        if negative_interval == interval:
            raise ValueError("negative_interval must differ from interval")

    # Find families that can realize this interval on at least two different starts
    family_to_starts: Dict[str, List[int]] = {}
    for family in instrument_families:
        starts = sorted(set(valid_interval_starts(interval, family)))
        if len(starts) >= 2:
            family_to_starts[family] = starts

    if not family_to_starts:
        raise ValueError(f"No instrument families have two or more starts for interval {interval}")

    family = rng.choice(list(family_to_starts.keys()))
    starts = family_to_starts[family]

    anchor_start = rng.choice(starts)
    anchor_target = anchor_start + interval

    positive_start_choices = [p for p in starts if p != anchor_start]
    positive_start = rng.choice(positive_start_choices)
    positive_target = positive_start + interval

    if negative_interval is None:
        candidate_intervals = []
        for delta in range(-MAX_INTERVAL, MAX_INTERVAL + 1):
            if delta == 0 or delta == interval:
                continue
            target_pitch = positive_start + delta
            if target_pitch < MIN_MIDI or target_pitch > MAX_MIDI:
                continue
            if family in pitch_family_to_ids.get(target_pitch, {}):
                candidate_intervals.append(delta)
        if not candidate_intervals:
            raise ValueError(f"No negative intervals available from start pitch {positive_start} in family {family} within MIDI {MIN_MIDI}-{MAX_MIDI}")
        negative_interval = rng.choice(candidate_intervals)
    else:
        target_pitch = positive_start + negative_interval
        if not (MIN_MIDI <= target_pitch <= MAX_MIDI and family in pitch_family_to_ids.get(target_pitch, {})):
            raise ValueError(
                f"negative_interval {negative_interval} from start {positive_start} missing for family {family} within MIDI {MIN_MIDI}-{MAX_MIDI}"
            )

    negative_target = positive_start + negative_interval

    triplet = {
        "anchor": {
            "family": family,
            "start_pitch": anchor_start,
            "target_pitch": anchor_target,
            "interval": interval,
            "start_id": choose_file_for_pitch_family(anchor_start, family, rng),
            "target_id": choose_file_for_pitch_family(anchor_target, family, rng),
        },
        "positive": {
            "family": family,
            "start_pitch": positive_start,
            "target_pitch": positive_target,
            "interval": interval,
            "start_id": choose_file_for_pitch_family(positive_start, family, rng),
            "target_id": choose_file_for_pitch_family(positive_target, family, rng),
        },
        "negative": {
            "family": family,
            "start_pitch": positive_start,  # same start as positive
            "target_pitch": negative_target,
            "interval": negative_interval,
            "start_id": None,  # filled below to guarantee same audio start
            "target_id": choose_file_for_pitch_family(negative_target, family, rng),
        },
    }

    # Force the negative to share the exact start audio with the positive
    triplet["negative"]["start_id"] = triplet["positive"]["start_id"]
    return triplet


def describe_triplet(triplet: Dict[str, Dict]) -> None:
    for role in ("anchor", "positive", "negative"):
        entry = triplet[role]
        print(
            f"{role.capitalize():<8}: family {entry['family']:<12} | start MIDI {entry['start_pitch']:>3} → target {entry['target_pitch']:>3} "
            f"(interval {entry['interval']:+d})"
        )


def load_triplet_audio(triplet: Dict[str, Dict]) -> Tuple[Dict[str, torch.Tensor], int]:
    """Return concatenated interval clips for each role."""
    clips: Dict[str, torch.Tensor] = {}
    sr_out = None
    for role in ("anchor", "positive", "negative"):
        clip, sr = make_interval_clip(triplet[role]["start_id"], triplet[role]["target_id"])
        if sr_out is None:
            sr_out = sr
        elif sr_out != sr:
            raise ValueError("Sample rate mismatch across clips")
        clips[role] = clip
    return clips, sr_out



In [94]:
# Build one example: major third up (+4 semitones)
triplet = build_interval_triplet(interval=4, rng_seed=42)
describe_triplet(triplet)


Anchor  : family brass        | start MIDI  40 → target  44 (interval +4)
Positive: family brass        | start MIDI  64 → target  68 (interval +4)
Negative: family brass        | start MIDI  64 → target  60 (interval -4)


In [ ]:
# Listen to the three interval examples (each clip is 2.0s total)
clips, sr = load_triplet_audio(triplet)

for role in ("anchor", "positive", "negative"):
    entry = triplet[role]
    duration_sec = clips[role].shape[-1] / sr
    print(f"{role.capitalize():<8} | family {entry['family']:<12} | interval {entry['interval']:+d} | clip length {duration_sec:.2f}s")
    display(Audio(clips[role].numpy(), rate=sr))


Anchor   | family brass        | interval +4 | clip length 2.00s


Positive | family brass        | interval +4 | clip length 2.00s


Negative | family brass        | interval -4 | clip length 2.00s


In [104]:

# make mel filter bank
mel_spec = torchaudio.transforms.MelSpectrogram(
    n_fft=1028,
)

def make_spectrogram(audio: torch.Tensor) -> torch.Tensor:
    # Add batch/channel dims for transform, then squeeze back to 2D time-frequency
    return mel_spec(audio.unsqueeze(0)).squeeze(0)

anchor_spec = make_spectrogram(clips["anchor"])
positive_spec = make_spectrogram(clips["positive"])
negative_spec = make_spectrogram(clips["negative"])


## Load pretrained models (matching the Mandarin tone notebook)

Load the same supervised/SSL checkpoints used in `dev_mandarin_tone_task.ipynb` for evaluating simple distance metrics (no contrastive loss) on the NSynth interval triplets.


In [107]:
import importlib
from lightning_scripts.utils import model_build_utils
importlib.reload(model_build_utils)
from lightning_scripts.utils.model_build_utils import get_model

# Device and model sample rate
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SR = 20_000

# Reuse the configs from the Mandarin tone notebook
ssl_lambda_0_config = Path("model_configs/kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_0e-01_audioset_only.yaml")
ssl_lambda_5_config = Path("model_configs/kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01_audioset_only.yaml")
word_config = Path("model_configs/supervised_models/word_kell2018_MatchedDataset_LARS.yaml")
audioset_config = Path("model_configs/supervised_models/audioset_kell2018_MatchedDataset_LARS.yaml")
multitask_config = Path("model_configs/supervised_models/kell2018_word_speaker_audioset_MatchedDataset_LARS.yaml")

unbal_audioset_config = Path("model_configs/supervised_models/kell2018_audioset_unbalanced_supervised.yaml")

models = {
    "ssl_lambda_0": get_model(ssl_lambda_0_config, layer_out="relu4"),
    "ssl_lambda_5": get_model(ssl_lambda_5_config, layer_out="relu4"),
    "word": get_model(word_config, supervised=True, layer_out="relu4"),
    "audioset": get_model(audioset_config, supervised=True, layer_out="relu4"),
    "multitask": get_model(multitask_config, supervised=True, layer_out="relu4"),
    "unbal_audioset": get_model(unbal_audioset_config, supervised=True, layer_out="relu4"),
}

for m in models.values():
    m.eval().to(DEVICE)

print(f"Loaded {len(models)} models on {DEVICE} for distance-based evaluation")


Loaded 6 models on cuda for distance-based evaluation


In [113]:
def encode_audio(model, audio_batch: torch.Tensor) -> torch.Tensor:
    model.eval()
    with torch.no_grad():
        z = model(audio_batch)
        if z.ndim > 2:
            z = z.flatten(start_dim=1)
    return z


def distance_metrics_on_triplet(model, clips: Dict[str, torch.Tensor], sr: int) -> Dict[str, float]:
    """Compute simple L2 distances between anchor-positive and anchor-negative."""
    waves = {}
    for k, v in clips.items():
        w = v
        if sr != MODEL_SR:
            w = torchaudio.functional.resample(w.unsqueeze(0), sr, MODEL_SR).squeeze(0)
        waves[k] = w.to(DEVICE)

    # Add channel dimension expected by models: (B, 1, T)
    pair_batch = torch.stack([waves["anchor"], waves["positive"]], dim=0).unsqueeze(1)
    neg_batch = waves["negative"].unsqueeze(0).unsqueeze(1)

    z_pair = encode_audio(model, pair_batch)
    z_anchor, z_pos = z_pair[0].unsqueeze(0), z_pair[1].unsqueeze(0)
    z_neg = encode_audio(model, neg_batch)

    pos_l2 = torch.norm(z_anchor - z_pos, p=2).item()
    neg_l2 = torch.norm(z_anchor - z_neg, p=2).item()
    pos_l2_sq = pos_l2 ** 2
    neg_l2_sq = neg_l2 ** 2

    return {
        "pos_l2": pos_l2,
        "neg_l2": neg_l2,
        "pos_l2_sq": pos_l2_sq,
        "neg_l2_sq": neg_l2_sq,
        "judgement_pos_lt_neg": int(pos_l2 < neg_l2),
    }



In [114]:
# Run simple distance metrics on the current triplet for each model
# Ensure you have run the triplet sampling cell above to define `triplet` and `clips`

results = []
for name, model in models.items():
    metrics = distance_metrics_on_triplet(model, clips, sr)
    results.append({
        "model": name,
        **metrics,
    })
    print(
        f"{name:12} | pos_l2 {metrics['pos_l2']:.4f} | neg_l2 {metrics['neg_l2']:.4f} | "
        f"pos_l2_sq {metrics['pos_l2_sq']:.4f} | neg_l2_sq {metrics['neg_l2_sq']:.4f} | "
        f"pos<neg {metrics['judgement_pos_lt_neg']}"
    )

results


ssl_lambda_0 | pos_l2 354.8386 | neg_l2 420.1270 | pos_l2_sq 125910.4484 | neg_l2_sq 176506.6567 | pos<neg 1
ssl_lambda_5 | pos_l2 326.0453 | neg_l2 390.5609 | pos_l2_sq 106305.5498 | neg_l2_sq 152537.8030 | pos<neg 1
word         | pos_l2 705.5375 | neg_l2 836.7702 | pos_l2_sq 497783.1295 | neg_l2_sq 700184.3720 | pos<neg 1
audioset     | pos_l2 408.7850 | neg_l2 407.9884 | pos_l2_sq 167105.1543 | neg_l2_sq 166454.5621 | pos<neg 0
multitask    | pos_l2 531.8986 | neg_l2 587.2617 | pos_l2_sq 282916.1426 | neg_l2_sq 344876.2546 | pos<neg 1
unbal_audioset | pos_l2 73.0237 | neg_l2 82.7175 | pos_l2_sq 5332.4637 | neg_l2_sq 6842.1871 | pos<neg 1


[{'model': 'ssl_lambda_0',
  'pos_l2': 354.838623046875,
  'neg_l2': 420.126953125,
  'pos_l2_sq': 125910.44840580225,
  'neg_l2_sq': 176506.65674209595,
  'judgement_pos_lt_neg': 1},
 {'model': 'ssl_lambda_5',
  'pos_l2': 326.0453186035156,
  'neg_l2': 390.5608825683594,
  'pos_l2_sq': 106305.54978326801,
  'neg_l2_sq': 152537.8029925758,
  'judgement_pos_lt_neg': 1},
 {'model': 'word',
  'pos_l2': 705.5374755859375,
  'neg_l2': 836.7702026367188,
  'pos_l2_sq': 497783.12945617735,
  'neg_l2_sq': 700184.3720206954,
  'judgement_pos_lt_neg': 1},
 {'model': 'audioset',
  'pos_l2': 408.78497314453125,
  'neg_l2': 407.9884338378906,
  'pos_l2_sq': 167105.15426877514,
  'neg_l2_sq': 166454.56214549486,
  'judgement_pos_lt_neg': 0},
 {'model': 'multitask',
  'pos_l2': 531.8986206054688,
  'neg_l2': 587.2616577148438,
  'pos_l2_sq': 282916.1426020004,
  'neg_l2_sq': 344876.2546219863,
  'judgement_pos_lt_neg': 1},
 {'model': 'unbal_audioset',
  'pos_l2': 73.02371978759766,
  'neg_l2': 82.717

In [ ]:
import itertools

class NsynthTripletDataset(torch.utils.data.Dataset):
    """On-the-fly NSynth interval triplets with resampling."""
    def __init__(self, n_examples: int = 300, interval_choices=None, seed: int = 0):
        self.n_examples = n_examples
        if interval_choices is None:
            # signed intervals within allowed bounds (exclude 0)
            pos = list(range(MIN_INTERVAL, MAX_INTERVAL + 1))
            neg = [-i for i in pos]
            self.interval_choices = pos + neg
        else:
            self.interval_choices = interval_choices
        self.seed = seed

    def __len__(self):
        return self.n_examples

    def __getitem__(self, idx):
        rng = random.Random(self.seed + idx)
        # Try a few times to sample a valid interval/family/start combo
        for _ in range(10):
            interval = rng.choice(self.interval_choices)
            try:
                triplet = build_interval_triplet(interval=interval, rng_seed=rng.randint(0, 1_000_000))
                clips, sr = load_triplet_audio(triplet)
                # resample to model SR and keep 1D (channel added later)
                if sr != MODEL_SR:
                    for k, v in clips.items():
                        clips[k] = torchaudio.functional.resample(v.unsqueeze(0), sr, MODEL_SR).squeeze(0)
                    sr = MODEL_SR
                return clips, sr, triplet
            except Exception:
                continue
        raise RuntimeError("Failed to sample a valid triplet after multiple attempts")


def nsynth_triplet_collate(batch):
    # batch of size 1 is expected; unpack
    clips, sr, triplet = batch[0]
    return clips, sr, triplet


triplet_ds = NsynthTripletDataset(n_examples=300, seed=123)
triplet_loader = torch.utils.data.DataLoader(triplet_ds, batch_size=1, shuffle=False, collate_fn=nsynth_triplet_collate)
print(f"Triplet dataset ready: {len(triplet_ds)} examples")


In [ ]:
# Evaluate distance metrics over the triplet loader (300 examples)
all_results = []
for batch_idx, (clips_b, sr_b, triplet_b) in enumerate(triplet_loader):
    # clips_b is a dict of length-1 tensors; unwrap
    clips = {k: v.squeeze(0) for k, v in clips_b.items()}
    sr_val = int(sr_b[0])
    for name, model in models.items():
        metrics = distance_metrics_on_triplet(model, clips, sr_val)
        metrics.update({
            "model": name,
            "batch_idx": batch_idx,
            "interval": triplet_b[0]["anchor"]["interval"],
            "family": triplet_b[0]["anchor"]["family"],
        })
        all_results.append(metrics)

import pandas as pd
results_df = pd.DataFrame(all_results)
print(results_df.head())
print("\nJudgement rate (pos<neg) per model:")
print(results_df.groupby("model")["judgement_pos_lt_neg"].mean())
